# Lab M — Model Context Protocol: scope design is security design

**Curriculum §5 · Track 5**

MCP standardises how agents reach tools/data. In a bank the hard part is not the protocol — it is **authorisation**: a broad scope hands an unpredictable model broad power. We build a small **FastMCP** server of core-banking tools, then wrap it in **per-role scopes** (teller / adviser / ops) and watch a deliberate privilege-escalation attempt get blocked.

> Colab can't easily run a stdio server + separate client, so we exercise the tools **in-process**. The scope logic — the actual lesson — is identical to production.

## ▶ Colab setup — run this cell first

1. Add your keys in Colab: **🔑 (left sidebar) → Secrets** → add `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `GROQ_API_KEY` (toggle *Notebook access* on).
2. Free signups: **Langfuse** → cloud.langfuse.com · **Groq** → console.groq.com
3. Run the cell. It installs deps and clones the shared `common/` package from GitHub.

> **If you see a `PIL._typing._Ink` import error:** run the cell, then **Runtime → Restart session**, then re-run. It's a Colab package clash, fixed by the Pillow upgrade + a restart.

In [ ]:
# --- Colab bootstrap (safe to re-run) ---
import os, sys, subprocess, pathlib
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    subprocess.run('pip install -q -U Pillow'.split())          # fix Colab PIL._typing._Ink clash
    subprocess.run('pip install -q langfuse ragas sentence-transformers faiss-cpu rank_bm25 langchain langchain-community langchain-groq langchain-text-splitters langgraph pypdf pdfplumber PyMuPDF pandas mcp'.split())

    # The repo is public — clone it to get the shared common/ package.
    REPO = pathlib.Path('/content/labpractice')
    if not (REPO/'common'/'harness.py').exists():
        subprocess.run(['git','clone','--depth','1',
                        'https://github.com/AICareerstack26/labpractice', str(REPO)])
    sys.path.insert(0, str(REPO))

    # Keys from Colab Secrets (🔑 sidebar) -> env vars that common/config.py reads.
    try:
        from google.colab import userdata
        for k in ['LANGFUSE_PUBLIC_KEY','LANGFUSE_SECRET_KEY','GROQ_API_KEY']:
            v = userdata.get(k)
            if v: os.environ[k] = v
        os.environ.setdefault('LANGFUSE_HOST','https://cloud.langfuse.com')
    except Exception as e:
        print('Secrets not set — running offline. (', e, ')')
else:
    sys.path.insert(0, str(pathlib.Path.cwd().parent))          # local fallback

print('Environment:', 'Colab' if IN_COLAB else 'Local')

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd().parent))              # make `common` importable locally
from common.config import *
from common.corpus import DOCS, current_docs, SUPERSEDED_IDS
from common.golden import GOLDEN
from common.obs import observe, span_meta, trace_meta, make_config, flush, ENABLED
from common.harness import evaluate, leaderboard
print('Langfuse tracing:', 'ON' if ENABLED else 'OFF (labs still run)')
print('AS_OF:', AS_OF, '| in force:', [d['id'] for d in current_docs()], '| superseded:', SUPERSEDED_IDS)

## 1 · A FastMCP server of core-banking tools

If the `mcp` SDK import fails on Colab, we fall back to a plain registry with the *same* tool functions — the authorisation lesson does not depend on the transport.

In [ ]:
try:
    from mcp.server.fastmcp import FastMCP
    mcp = FastMCP("meridian-core")
    HAVE_MCP = True
except Exception as e:
    print("mcp SDK unavailable, using in-process registry (", type(e).__name__, ")")
    HAVE_MCP = False
    class _Reg:
        def __init__(self): self.fns = {}
        def tool(self):
            def deco(f): self.fns[f.__name__] = f; return f
            return deco
    mcp = _Reg()

ACCOUNTS = {"ACC-1001": {"name":"J. Okafor","balance":4210.55,"status":"active"},
            "ACC-2002": {"name":"R. Silva","balance":88.20,"status":"frozen"}}

@mcp.tool()
def get_account(account_id: str) -> dict:
    """Read basic account details."""
    return ACCOUNTS.get(account_id, {"error":"not found"})

@mcp.tool()
def list_transactions(account_id: str, limit: int = 3) -> list:
    """List recent transactions for an account."""
    return [{"id":f"T{account_id[-2:]}{i}","amt":-i*12.5} for i in range(1, limit+1)]

@mcp.tool()
def adjust_limit(account_id: str, new_limit: int) -> dict:
    """OPS-ONLY: change an account's credit limit (privileged, irreversible)."""
    ACCOUNTS.setdefault(account_id, {})["limit"] = new_limit
    return {"account_id":account_id, "limit":new_limit, "status":"updated"}

REGISTRY = {f.__name__: f for f in [get_account, list_transactions, adjust_limit]}
print("tools:", list(REGISTRY))

## 2 · Least privilege — scopes per role

The server exposes everything; the **client's scope** decides what it may call. This is the whole game: `adjust_limit` is destructive, so only the `ops` role may reach it.

In [ ]:
SCOPES = {
  "teller":  {"get_account", "list_transactions"},
  "adviser": {"get_account", "list_transactions"},
  "ops":     {"get_account", "list_transactions", "adjust_limit"},
}

class ScopedClient:
    def __init__(self, role): self.role = role; self.allowed = SCOPES[role]
    def call(self, tool, **args):
        if tool not in self.allowed:
            raise PermissionError(f"role '{self.role}' is not scoped for '{tool}'")
        return REGISTRY[tool](**args)

teller = ScopedClient("teller")
ops    = ScopedClient("ops")
print("teller get_account :", teller.call("get_account", account_id="ACC-1001"))
print("ops    adjust_limit:", ops.call("adjust_limit", account_id="ACC-1001", new_limit=5000))

## 3 · The escalation attempt — blocked, and audited

In [ ]:
AUDIT = []
def attempt(client, tool, **args):
    try:
        r = client.call(tool, **args); ok = True
    except PermissionError as e:
        r = str(e); ok = False
    AUDIT.append(dict(role=client.role, tool=tool, allowed=ok))
    print(f"[{client.role:7s}] {tool:16s} -> {'OK' if ok else 'DENIED'}: {r}")

attempt(teller, "get_account",  account_id="ACC-1001")
attempt(teller, "adjust_limit", account_id="ACC-1001", new_limit=999999)   # privilege escalation
attempt(ops,    "adjust_limit", account_id="ACC-2002", new_limit=3000)
import pandas as pd; pd.DataFrame(AUDIT)

## 4 · What you should conclude

- MCP makes integration *reusable*, but the security boundary is the **scope**, not the protocol. A server that exposes `adjust_limit` to every client has already lost — regardless of how clean the MCP wiring is.
- **Scope design = security design.** Model the smallest capability each role needs, deny by default, and **audit every call** (the `AUDIT` table is your regulator-grade trail).
- In production the same tools sit behind a real MCP transport (stdio/HTTP) with per-client identity — but the authorisation model you just wrote is exactly what ships.

> **You've now built the full stack:** observability → RAG → retrieval/rerank → inference → fine-tuning → agent → MCP, all on the Meridian Bank scenario, all measured. That is a portfolio.